# Command-Line Interface (`craft-support`)

> Command-line entry point (`craft-support`) for the `solveitcrafts` support workflow.

Supports both Google Drive (`--store drive`, default) and offline local filesystem (`--store local`) backends across all subcommands: `serve`, `tui`, `list`, `show`, `note`, `assign`, `merge`, `close`, and `teardown`.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import argparse, asyncio, inspect, json
from fastcore.utils import *

from craft_support.drive import DEFAULT_ROOT_FOLDER, DriveTicketStore, LocalTicketStore, TicketStore
from craft_support.models import VALID_STATUSES
from craft_support.orchestrator import FastGitAdapter, GhApiAdapter, ReviewOrchestrator
from craft_support.solveit_cdp import FastCDPReviewAppProvisioner

## Store & Orchestrator Builders

In [ ]:
#| export
async def _build_store(args: argparse.Namespace) -> TicketStore:
    if args.store == "local": return LocalTicketStore(Path(args.local_dir))
    return await DriveTicketStore.from_oauth(account=args.account, root_folder_name=args.drive_folder)

def _build_orchestrator(store: TicketStore, args: argparse.Namespace) -> ReviewOrchestrator:
    repo_dir = Path(args.repo_dir)
    dist_dir = repo_dir/"dist"
    return ReviewOrchestrator(store=store, git=FastGitAdapter(repo_dir, push_enabled=not args.no_push),
        gh=GhApiAdapter(owner=args.gh_owner, repo=args.gh_repo), provisioner=FastCDPReviewAppProvisioner(auth_file=Path(args.auth_file)),
        assets_dir=dist_dir if dist_dir.is_dir() else None)

## Subcommand Handlers

In [ ]:
#| export
async def _cmd_list(args: argparse.Namespace) -> int:
    store = await _build_store(args)
    tickets = await store.list_tickets(status=args.status, refresh=args.refresh)
    if args.json:
        print(json.dumps([t.to_dict() for t in tickets], indent=2))
        return 0
    if not tickets:
        print("No tickets found.")
        return 0
    for t in tickets:
        pr = f"PR#{t.pr_number}" if t.pr_number else "-"
        print(f"{t.id}  [{t.status:9s}]  {t.title[:48]:48s}  {pr:8s}  {t.review_app_id or '-'}")
    return 0

async def _cmd_show(args: argparse.Namespace) -> int:
    store = await _build_store(args)
    print((await store.get_ticket(args.ticket_id)).to_json())
    return 0

async def _cmd_note(args: argparse.Namespace) -> int:
    store = await _build_store(args)
    t = await store.add_triage_note(args.ticket_id, args.text, author=args.author)
    print(f"Updated {t.id} ({t.status}), {len(t.triage_notes)} note(s).")
    return 0

### Review Lifecycle, Server & TUI Commands

In [ ]:
#| export
async def _cmd_assign(args: argparse.Namespace) -> int:
    orch = _build_orchestrator(await _build_store(args), args)
    res = await orch.assign_tickets(args.ticket_ids, base_branch=args.base_branch, existing_branch=args.existing_branch,
        branch_name=args.branch, title=args.title)
    print(f"{'Reused' if res.reused else 'Provisioned'} {res.group_id}: "
        f"branch={res.branch} pr=#{res.pr.number} ({res.pr.url}) app={res.review_app.review_app_id} ({res.review_app.url})")
    return 0

async def _cmd_complete(args: argparse.Namespace, action: str) -> int:
    orch = _build_orchestrator(await _build_store(args), args)
    res = await orch.complete_review(
        args.ticket_ids, action=action, merge_method=getattr(args, "merge_method", "squash"), delete_branch=not args.keep_branch)
    print(f"{action.title()}d {', '.join(args.ticket_ids)}; tore down {res['torndown_apps']}")
    return 0

async def _cmd_teardown(args: argparse.Namespace) -> int:
    ok = await FastCDPReviewAppProvisioner(auth_file=Path(args.auth_file)).teardown(args.review_app_id)
    print(f"Tore down {args.review_app_id}" if ok else f"{args.review_app_id} was not active.")
    return 0

def _cmd_serve(args: argparse.Namespace) -> int:
    import uvicorn
    from craft_support.server import create_app
    uvicorn.run(create_app(asyncio.run(_build_store(args))), host=args.host, port=args.port)
    return 0

def _cmd_tui(args: argparse.Namespace) -> int:
    from craft_support.tui import TriageApp
    store = asyncio.run(_build_store(args))
    orch = _build_orchestrator(store, args) if args.gh_owner and args.gh_repo else None
    TriageApp(store=store, orchestrator=orch).run()
    return 0

## Argument Parser & `main`

In [ ]:
#| export
_GLOBAL_OPTS = [
    ("--store", dict(choices=["drive", "local"], default=os.environ.get("CRAFT_SUPPORT_STORE", "drive"))),
    ("--account", dict(default=os.environ.get("CRAFT_SUPPORT_ACCOUNT"))),
    ("--drive-folder", dict(default=os.environ.get("CRAFT_SUPPORT_DRIVE_FOLDER", DEFAULT_ROOT_FOLDER))),
    ("--local-dir", dict(default=os.environ.get("CRAFT_SUPPORT_LOCAL_DIR", ".craft-tickets"))),
    ("--repo-dir", dict(default=os.environ.get("CRAFT_SUPPORT_REPO_DIR", "."))),
    ("--gh-owner", dict(default=os.environ.get("CRAFT_SUPPORT_GH_OWNER", "abiancar"))),
    ("--gh-repo", dict(default=os.environ.get("CRAFT_SUPPORT_GH_REPO", "solve-it-crafts"))),
    ("--auth-file", dict(default=os.environ.get("CRAFT_SUPPORT_AUTH_FILE", ".auth/solveit-state.json"))),
    ("--no-push", dict(action="store_true"))]

_SUBCOMMANDS = [
    ("serve", "Run the FastHTML bug-capture ingestion server.", _cmd_serve, [
        ("--host", dict(default="127.0.0.1")), ("--port", dict(type=int, default=8791)),
    ]),
    ("tui", "Launch the interactive terminal triage UI.", _cmd_tui, []),
    ("list", "List tickets.", _cmd_list, [
        ("--status", dict(choices=list(VALID_STATUSES))),
        ("--refresh", dict(action="store_true")), ("--json", dict(action="store_true")),
    ]),
    ("show", "Print ticket.json for a ticket ID.", _cmd_show, [("ticket_id", {})]),
    ("note", "Append a triage note to a ticket.", _cmd_note, [
        ("ticket_id", {}), ("text", {}), ("--author", dict(default="operator")),
    ]),
    ("assign", "Assign ticket(s) to a branch, PR, and Solveit review app.", _cmd_assign, [
        ("ticket_ids", dict(nargs="+")), ("--base-branch", dict(default="main")),
        ("--existing-branch", {}), ("--branch", {}), ("--title", {}),
    ]),
    ("merge", "Merge PR, tear down Solveit review app, and mark tickets merged.", partial(_cmd_complete, action="merge"), [
        ("ticket_ids", dict(nargs="+")),
        ("--merge-method", dict(choices=["merge", "squash", "rebase"], default="squash")),
        ("--keep-branch", dict(action="store_true")),
    ]),
    ("close", "Close PR, tear down Solveit review app, and mark tickets closed.", partial(_cmd_complete, action="close"), [
        ("ticket_ids", dict(nargs="+")), ("--keep-branch", dict(action="store_true")),
    ]),
    ("teardown", "Tear down a Solveit review app by ID.", _cmd_teardown, [("review_app_id", {})])]

def build_parser() -> argparse.ArgumentParser:
    "Build the `craft-support` CLI argument parser."
    p = argparse.ArgumentParser(prog="craft-support", description="Bug-to-review-app support workflow for solveitcrafts")
    for flag, kw in _GLOBAL_OPTS: p.add_argument(flag, **kw)
    sub = p.add_subparsers(dest="command", required=True)
    for name, help_txt, fn, opts in _SUBCOMMANDS:
        sp = sub.add_parser(name, help=help_txt)
        for flag, kw in opts: sp.add_argument(flag, **kw)
        sp.set_defaults(func=fn)
    return p

def main(
    argv: list[str] | None = None, # CLI argument list (defaults to `sys.argv[1:]`)
) -> int:
    "Entry point for the `craft-support` console script."
    args = build_parser().parse_args(argv)
    res = args.func(args)
    return asyncio.run(res) if inspect.iscoroutine(res) else res

In [ ]:
show_doc(build_parser)
show_doc(main)

In [ ]:
import concurrent.futures, contextlib, io, tempfile
from fastcore.test import test_eq

Verify parser coverage across all subcommands and run `list`, `note`, and `show` against a temporary `LocalTicketStore`.

In [ ]:
parser = build_parser()
test_eq(parser.parse_args(["serve", "--port", "9000"]).port, 9000)
test_eq(parser.parse_args(["tui"]).command, "tui")
test_eq(parser.parse_args(["--store", "local", "list", "--status", "new", "--json"]).json, True)
test_eq(parser.parse_args(["assign", "TKT-1", "TKT-2", "--base-branch", "feat/x"]).ticket_ids, ["TKT-1", "TKT-2"])
test_eq(parser.parse_args(["merge", "TKT-1", "--merge-method", "rebase", "--keep-branch"]).keep_branch, True)
test_eq(parser.parse_args(["close", "TKT-1"]).command, "close")
test_eq(parser.parse_args(["teardown", "review-1"]).review_app_id, "review-1")

def _run_cli(argv: list[str]) -> tuple[int, str]:
    def _invoke():
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf): rc = main(argv)
        return rc, buf.getvalue()
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex: return ex.submit(_invoke).result()

with tempfile.TemporaryDirectory() as td:
    local_dir = str(Path(td)/"tickets")
    t, _ = await LocalTicketStore(local_dir).create_ticket(title="CLI smoke ticket", description="Test CLI commands")

    rc, out = _run_cli(["--store", "local", "--local-dir", local_dir, "list", "--json"])
    listed = loads(out)
    test_eq((rc, len(listed), listed[0]["id"]), (0, 1, t.id))

    rc, out = _run_cli(["--store", "local", "--local-dir", local_dir, "note", t.id, "Triaged via CLI"])
    test_eq(rc, 0)
    assert "triaged" in out

    rc, out = _run_cli(["--store", "local", "--local-dir", local_dir, "show", t.id])
    test_eq((rc, loads(out)["status"]), (0, "triaged"))